In [1]:
"""
experiments.py  (dynamic / vehicular version)

Scenario 1 — vary SNR
Scenario 2 — vary number of STAR-RIS elements N
Scenario 3 — vary car speed (new: replaces static K sweep)
Scenario 4 — vary number of cars K
"""

import numpy as np
import pandas as pd
from simulator import DEFAULT_PARAMS, init_cars
from methods import method_classical, method_qaoa, method_hybrid
import copy


def _run_one_trial(p):
    """
    Create fresh cars, run all three methods, return results.
    Cars are re-created per trial so mobility is independent.
    """
    cars1 = init_cars(p)
    cars2 = init_cars(p)   # separate car objects per method
    cars3 = init_cars(p)   # so they don't share state

    # copy initial positions so all three methods start from same snapshot
    for ca, cb, cc in zip(cars1, cars2, cars3):
        cb.pos = ca.pos.copy(); cb.vel = ca.vel.copy()
        cc.pos = ca.pos.copy(); cc.vel = ca.vel.copy()

    r1 = method_classical(cars1, p)
    r2 = method_qaoa(cars2, p)
    r3 = method_hybrid(cars3, p)
    return r1, r2, r3


def _aggregate(records):
    keys = ['sum_rate', 'iterations', 'energy_norm', 'time_s']
    out  = {}
    for k in keys:
        vals = [r[k] for r in records]
        out[k+'_mean'] = float(np.mean(vals))
        out[k+'_std']  = float(np.std(vals))
    return out


def scenario_snr(snr_db_range=range(0, 31, 5), n_trials=20,
                 base_params=None):
    p0 = (base_params or DEFAULT_PARAMS).copy()
    rows = []
    for snr_db in snr_db_range:
        p = p0.copy()
        p['sigma2'] = p['P_max'] / (10 ** (snr_db / 10))
        print(f"  SNR={snr_db} dB", end='', flush=True)
        r1s, r2s, r3s = [], [], []
        for _ in range(n_trials):
            r1, r2, r3 = _run_one_trial(p)
            r1s.append(r1); r2s.append(r2); r3s.append(r3)
            print('.', end='', flush=True)
        print()
        for name, recs in [('classical',r1s),('qaoa',r2s),('hybrid',r3s)]:
            agg = _aggregate(recs)
            agg.update(snr_db=snr_db, method=name)
            rows.append(agg)
    return pd.DataFrame(rows)


def scenario_N(N_values=(16, 32, 64), n_trials=20, base_params=None):
    p0 = (base_params or DEFAULT_PARAMS).copy()
    p0['sigma2'] = p0['P_max'] / (10 ** (10 / 10))
    rows = []
    for N in N_values:
        p = p0.copy(); p['N'] = N
        print(f"  N={N}", end='', flush=True)
        r1s, r2s, r3s = [], [], []
        for _ in range(n_trials):
            r1, r2, r3 = _run_one_trial(p)
            r1s.append(r1); r2s.append(r2); r3s.append(r3)
            print('.', end='', flush=True)
        print()
        for name, recs in [('classical',r1s),('qaoa',r2s),('hybrid',r3s)]:
            agg = _aggregate(recs)
            agg.update(N=N, method=name)
            rows.append(agg)
    return pd.DataFrame(rows)


def scenario_speed(speed_values=(5, 10, 20, 30), n_trials=20,
                   base_params=None):
    """
    New scenario: vary car speed (v_min = v_max = v so all cars go same speed).
    Shows how fast channel variation affects each method.
    """
    p0 = (base_params or DEFAULT_PARAMS).copy()
    p0['sigma2'] = p0['P_max'] / (10 ** (10 / 10))
    rows = []
    for v in speed_values:
        p = p0.copy(); p['v_min'] = v; p['v_max'] = v
        print(f"  speed={v} m/s", end='', flush=True)
        r1s, r2s, r3s = [], [], []
        for _ in range(n_trials):
            r1, r2, r3 = _run_one_trial(p)
            r1s.append(r1); r2s.append(r2); r3s.append(r3)
            print('.', end='', flush=True)
        print()
        for name, recs in [('classical',r1s),('qaoa',r2s),('hybrid',r3s)]:
            agg = _aggregate(recs)
            agg.update(speed_ms=v, method=name)
            rows.append(agg)
    return pd.DataFrame(rows)


def scenario_K(K_values=((2,2),(4,4),(8,8)), n_trials=20,
               base_params=None):
    p0 = (base_params or DEFAULT_PARAMS).copy()
    p0['sigma2'] = p0['P_max'] / (10 ** (10 / 10))
    rows = []
    for Kr, Kt in K_values:
        p = p0.copy(); p['Kr'] = Kr; p['Kt'] = Kt
        print(f"  K={Kr+Kt}", end='', flush=True)
        r1s, r2s, r3s = [], [], []
        for _ in range(n_trials):
            r1, r2, r3 = _run_one_trial(p)
            r1s.append(r1); r2s.append(r2); r3s.append(r3)
            print('.', end='', flush=True)
        print()
        for name, recs in [('classical',r1s),('qaoa',r2s),('hybrid',r3s)]:
            agg = _aggregate(recs)
            agg.update(K=Kr+Kt, method=name)
            rows.append(agg)
    return pd.DataFrame(rows)

ModuleNotFoundError: No module named 'simulator'